# MACS 30111 — M7 Pandas, day 1: in-class notebook
Slides: `M7_Pandas_30111.pptx`, slides 1–25 · Textbook: §4.3.1–4.3.4 · Docs: https://pandas.pydata.org/docs/

**How to use this notebook:** run the **Example** cells as we go. In each **Try it** cell, predict the answer first, then write the code.

**Data:** the 2015 NYC Street Tree Census (683,788 trees). Put the CSV in a folder called `data/` next to this notebook.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 12)   # wide tables: show 12 columns, '...' for the rest

## 1. Series and DataFrames (slides 8–13)
A **Series** is one labeled column: values plus an **index** (the labels). A **DataFrame** is a table: several Series that share the same index.

### Example: build a Series (slide 10)

In [ ]:
# from a list, plus a list of labels
s = pd.Series([90, 95, 90], index=["Rory", "Lorelai", "Luke"])
s

In [ ]:
# from a dictionary: the keys become the index
s = pd.Series({"Rory": 90, "Lorelai": 95, "Luke": 90})
s

### Example: build a DataFrame (slides 11–13)

In [ ]:
# dictionary of lists: the keys become the COLUMNS
data = {"PA1": [90, 95, 90], "PA2": [88, 90, 75]}
df = pd.DataFrame(data, index=["Rory", "Lorelai", "Luke"])
df

In [ ]:
# dictionary of dictionaries: outer keys -> columns, inner keys -> rows. Same table!
data = {"PA1": {"Rory": 90, "Lorelai": 95, "Luke": 90},
        "PA2": {"Rory": 88, "Lorelai": 90, "Luke": 75}}
pd.DataFrame(data)

In [ ]:
# from a NumPy array: you supply the row and column labels
rng = np.random.default_rng(13)
arr = rng.uniform(0, 1, (3, 2))
pd.DataFrame(arr, index=["Rory", "Lorelai", "Luke"], columns=["PA1", "PA2"])

In [ ]:
# first look: what's in it?
print(df.shape)    # (rows, columns)
print(df.dtypes)   # one type per column
df["PA1"]          # one column -> a Series

### Try it 1.1
Make a DataFrame called `scores` for four students, **Dwight**, **Angela**, **Pam** and **Jim**, with columns `SE1` = 8, 10, 9, 7 and `SE2` = 7, 9, 10, 8. Then:
1. Print its shape.
2. Get the `SE2` column. What type is it?
3. Compute the class average on `SE1` (`.mean()` works on a Series, just like NumPy).

In [ ]:
# your code here


## 2. Load real data (slides 14–15, 20)
`pd.read_csv` reads a file into a DataFrame. This one is big, so it takes a few seconds.

**Questions we'll be able to answer by the end of the module (slide 20):**
1. How many different species are planted as street trees in New York?
2. What are the five most common street tree species?
3. What is the most common street tree species in Brooklyn?
4. What percentage of the street trees in Queens are dead or in poor health? **← today**
5. How does street tree health differ by borough?

### Example: load and look

In [ ]:
trees = pd.read_csv("data/2015_Street_Tree_Census_sample.csv")
trees.shape

In [ ]:
trees.head()

In [ ]:
trees.dtypes.head(10)   # 'object' = text

### Try it 2.1
1. Show the **last 3** rows.
2. How many columns are there? (Use `.shape`, not counting by hand.)
3. Print the column names. Which column holds the borough?

In [ ]:
# your code here


## 3. Missing data (slide 19)
`.isna()` gives True where a value is missing (`NaN`). Because True counts as 1, `.isna().sum()` counts the missing values in each column.

In [ ]:
trees.isna().sum().sort_values(ascending=False).head(8)

### Try it 3.1
**Predict first:** why would a tree have no `health` value?

Then check: how many trees have a missing `health`, and what is their `status`? *(Hint: filter to the trees with missing health, then look at the `status` column; `.value_counts()` counts each value.)*

In [ ]:
# your code here


## 4. Selecting rows and columns (slides 21–24)
- `iloc[...]` selects by **position**: row 0, 1, 2, ...
- `loc[...]` selects by **label**: whatever is in the index.

### Example: predict, then run
`trees.head()` shows that the first row is tree **180683**. **Predict:** do these two give the same tree?

In [ ]:
trees.iloc[0][["tree_id", "borough", "spc_common"]]

In [ ]:
trees.loc[180683][["tree_id", "borough", "spc_common"]]

They don't. Right now the index is just row numbers 0, 1, 2, ..., so `loc[180683]` means "the row **labeled** 180683", which is row 180,683, not tree 180683.

### Example: make the tree id the index
Now the labels *are* tree ids, so `loc` finds the tree by its id.

In [ ]:
trees = pd.read_csv("data/2015_Street_Tree_Census_sample.csv", index_col="tree_id")
trees.loc[180683][["borough", "spc_common", "health"]]

### Example: one column vs. a list of columns (slide 23)
**Predict** the type of each before running.

In [ ]:
print(type(trees["health"]))      # one name  -> a Series
print(type(trees[["health"]]))    # a list    -> a DataFrame
trees[["health", "borough"]].head()

In [ ]:
# rows and columns together (slide 24)
print(trees.iloc[:5][["health", "borough"]])                     # first 5 rows, by position
trees.loc[[180683, 200540, 204026], ["health", "borough"]]       # three trees, by id

### Try it 4.1
1. Get the tree in the **10th row** (careful: positions start at 0).
2. Get tree **192755** (the tree id). What species is it, and what borough?
3. Get the `spc_common` and `tree_dbh` columns for the first 5 trees.

In [ ]:
# your code here


## 5. Filtering with Boolean masks (slide 25)
Same idea as NumPy: a comparison gives True/False for every row, and `trees[mask]` keeps the True rows. Combine conditions with `&` (and) and `|` (or), with **parentheses around each one**.

Name your mask `mask` or something descriptive, not `filter` (that's a built-in Python function).

In [ ]:
is_dead = trees["status"] == "Dead"
print(is_dead.head())        # one True/False per tree
print(is_dead.sum())         # 13,961 dead trees
trees[is_dead][["status", "health", "borough"]].head()

In [ ]:
# the opposite: keep everything that is NOT dead
trees[trees["status"] != "Dead"].shape

### Try it 5.1: motivating question 4
**What percentage of the street trees in Queens are dead or in poor health?**

Steps: (1) keep only the Queens trees; (2) build a mask for "dead **or** poor"; (3) divide.

*Think about it:* dead trees have `health = NaN`. What would you miss if you only checked `health == "Poor"`?

In [ ]:
# your code here


### Try it 5.2
How many **red maples** in **Brooklyn** are in **Good** health? *(Three conditions, all with `&`.)*

In [ ]:
# your code here


## 6. Challenge (if you finish early): motivating questions 1–3
You'll need two new tools: `.nunique()` (how many different values) and `.value_counts()` (how many of each value, largest first).

In [ ]:
# 1. How many different species?


In [ ]:
# 2. The five most common species


In [ ]:
# 3. The most common species in Brooklyn
